<a href="https://colab.research.google.com/github/marcosfelipp/aulas-neuro-ai/blob/main/aula-01-picos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/marcosfelipp/aulas-neuro-ai/main/aula-01-picos.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Aula 1 · Detectando disparos num neurônio

Variáveis, condicionais e laços de repetição, num problema de verdade.

Rode as células em ordem. Onde tiver `TODO`.
Depois de cada exercício tem um `assert`: se passar, imprime `ok`.

## 1 · O dado

`sinal_neuronio.csv` — 10 s de registro de um neurônio, 2000 medições por segundo
(20 000 linhas). Duas colunas: `tempo_s` e `sinal_uv` (voltagem em microvolts).

Um **disparo** é uma deflexão grande e rápida no meio do ruído de fundo. Detectar é
perguntar, amostra por amostra: *passou do limiar?*

Sinal sintético, gerado por `gerar_dataset.py`. A célula abaixo baixa o arquivo do
GitHub — rode ela primeiro, no Colab ou na sua máquina.

In [ ]:
# ============ Baixa o dataset do GitHub ============

import urllib.request

URL = "https://raw.githubusercontent.com/marcosfelipp/aulas-neuro-ai/main/sinal_neuronio.csv"
urllib.request.urlretrieve(URL, "sinal_neuronio.csv")

print("sinal_neuronio.csv baixado")

In [ ]:
# ============ ESQUELETO — já vem pronto ============

tempo = []    # instantes, em segundos
sinal = []    # voltagens, em microvolts

with open("sinal_neuronio.csv", "r") as arquivo:
    arquivo.readline()                      # descarta o cabeçalho
    for linha in arquivo:
        partes = linha.strip().split(",")   # "0.0005,-11.19" -> ["0.0005", "-11.19"]
        tempo.append(float(partes[0]))      # float(): o que vem do arquivo é TEXTO
        sinal.append(float(partes[1]))

print("amostras:", len(sinal))
print("duração :", tempo[-1], "s")
print("mínimo  :", min(sinal), "uV")
print("máximo  :", max(sinal), "uV")

Ruído entre ±30 µV, picos acima de 100. É essa distância que torna o disparo detectável.

## 2 · Aquecimento — o laço `for`

O objetivo desse código é achar o maior valor do registro sem usar a função `max()`. Complete o comando for do `for` e o if dentro dele.


In [ ]:
# TODO: guarde o maior valor do sinal e o instante em que ele aconteceu

maior = 0
instante = 0

for ...:
    if ...:
        maior = ...
        instante = ...

print(f"maior valor: {maior:.2f} uV  em t = {instante:.4f} s")

In [ ]:
assert abs(maior - 107.05) < 1e-6, f"esperado 107.05 uV, veio {maior}"
assert abs(instante - 5.7375) < 1e-6, f"esperado t = 5.7375 s, veio {instante}"
print("ok")

## 3 · Problema 1 — quando houve um disparo?

O ruído tem desvio-padrão de ~9 µV. A convenção é cortar em 4–5 desvios: **limiar = 40 µV**.

In [ ]:
limiar = 40.0

# TODO: complete a condição (a amostra passou do limiar?)

for i in range(4000):                  # só os 2 primeiros segundos
    if ...:
        print(f"Spike! em t = {tempo[i]:.4f} s   ({sinal[i]:.1f} uV)")

## 4 · Problema 2 — contar

In [ ]:
# TODO: conte quantas amostras passaram do limiar

n_amostras_acima = 0

for i in range(len(sinal)):
    if ...:
        n_amostras_acima += 1

print("amostras acima do limiar:", n_amostras_acima)

In [ ]:
assert n_amostras_acima == 123, f"esperado 123, veio {n_amostras_acima}"
print("ok")

## 5 · Problema 3 — taxa de disparo

Disparos por segundo, em hertz.

In [ ]:
# TODO: duração do registro, e depois a taxa

duracao = ...
n_disparos = ...
taxa = ...

print(f"{duracao:.2f} s · {n_disparos} disparos · {taxa:.2f}")

In [ ]:
assert abs(duracao - 9.9995) < 1e-3, f"duração esperada ~10 s, veio {duracao}"
print("ok")

## 6 · Problema 4 — o gráfico

A função abaixo já está pronta — você só precisa **chamá-la**. Leia a assinatura:
ela diz, em ordem, o que a função espera receber.

(Escrever função é a próxima aula. Aqui basta usar.)

In [ ]:
# ============ ESQUELETO — já vem pronto ============

import matplotlib.pyplot as plt


def plotar_sinal(tempo, sinal, limiar):
    """tempo e sinal: as listas. limiar: o número."""
    figura, (cima, baixo) = plt.subplots(2, 1, figsize=(13, 6))

    tempos_disparos = []

    for i in range(1, len(sinal)):
        if sinal[i] > limiar and sinal[i - 1] <= limiar:
            tempos_disparos.append(tempo[i])

    for eixo, (t_ini, t_fim), titulo in [
        (cima,  (tempo[0], tempo[-1]), "Registro completo"),
        (baixo, (1.0, 2.0),            "Ampliação: 1 a 2 segundos"),
    ]:
        eixo.plot(tempo, sinal, linewidth=0.7, color="#3f6fb5")
        eixo.axhline(limiar, color="#ff7a3d", linestyle="--", linewidth=1.4,
                     label=f"limiar = {limiar:.0f} µV")
        marcados = [t for t in tempos_disparos if t_ini <= t <= t_fim]
        eixo.plot(marcados, [max(sinal) * 1.12] * len(marcados),
                  "v", color="#c4562a", markersize=6,
                  label=f"disparos ({len(marcados)})")
        eixo.set_xlim(t_ini, t_fim)
        eixo.set_ylim(min(sinal) * 1.15, max(sinal) * 1.45)
        eixo.set_title(titulo, fontsize=11)
        eixo.set_ylabel("µV")
        eixo.legend(loc="upper right", fontsize=8)

    baixo.set_xlabel("tempo (s)")
    figura.tight_layout()
    plt.show()

In [ ]:
# TODO: passe as três variáveis, na ordem da assinatura

plotar_sinal(..., ..., ...)